# 02. Extention of Edgar's Study

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     02-extention-of-edgar-s-study                      ║
# ║ Description:  Non-variable CpGs - Edgar's Extention              ║
# ║ Dataset(s):   GSE69914, GSE225845, G287331                       ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 26-Gen-2026 | Python 3.11.13                               ║
# ╚══════════════════════════════════════════════════════════════════╝


### Libraries

In [ ]:
# Latex
!sudo apt-get update -qq
!sudo apt-get install -y texlive-latex-extra texlive-fonts-recommended dvipng cm-super


In [ ]:
# Import
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import median_abs_deviation
import warnings
from scipy.stats import mannwhitneyu
from scipy.stats import gaussian_kde
from __future__ import annotations
import os, re, math
from typing import Iterable, Set, Dict, Tuple, Sequence, List
from polars import selectors as cs
import polars as pl
from itertools import combinations
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.cm as cm
from scipy.stats import spearmanr
from matplotlib_venn import venn3
from matplotlib.patches import Patch
from pathlib import Path

%config HistoryManager.enabled = False
warnings.filterwarnings('ignore')


In [ ]:
# THESIS STYLE FOR PLOT
def apply_thesis_style(use_tex: bool = True,
                       legend_position: str = "upper right",
                       legend_outside: bool = False):
    """
    Applies a uniform style (Matplotlib + Seaborn) consistent with LaTeX
    and defines `place_legend()` to position legends consistently.

    Parameters
    ----------
    use_tex : bool
        If True, enables LaTeX text rendering (requires TeX installed).
    legend_position : str
        DEFAULT position for legends: 'upper right', 'upper left',
        'lower right', 'lower left', 'center', 'best' (also accepts 'top/bottom').
    legend_outside : bool
        If True, the default legend is outside the plot (to the right).
    """

    # --- Normalize default position ---
    def _normalize_pos(pos: str) -> str:
        if not isinstance(pos, str):
            return "upper right"
        key = pos.strip().lower().replace("top", "upper").replace("bottom", "lower")
        mapping = {
            "upper right": "upper right",
            "upper left":  "upper left",
            "lower right": "lower right",
            "lower left":  "lower left",
            "center":      "center",
            "best":        "best",
        }
        return mapping.get(key, "upper right")

    _default_loc = _normalize_pos(legend_position)

    # --- Seaborn theme + rcParams consistent with thesis ---
    sns.set_theme(style="whitegrid", context="notebook")
    mpl.rcParams.update({
        # Typography
        "font.family": "serif",
        "font.serif": ["Computer Modern Roman", "Latin Modern Roman", "Times New Roman"],
        "mathtext.fontset": "cm",
        "text.usetex": bool(use_tex),

        # Sizes
        "figure.figsize": (6.8, 4.5),
        "font.size": 8.5,          # generic text (plt.text, etc.)
        "axes.labelsize": 10.5,    # axis labels
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9.5,
        "legend.title_fontsize": 9.5,

        # Axes look
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.facecolor": "white",
        "axes.edgecolor": "#D0D0D0",
        "axes.linewidth": 0.8,

        # Legend style (default; can be overridden by place_legend)
        "legend.frameon": True,
        "legend.facecolor": "white",
        "legend.edgecolor": "#D0D0D0",
        "legend.loc": _default_loc,
        "legend.framealpha": 1.0,
        "legend.handlelength": 1.8,
        "legend.handletextpad": 0.6,
        "legend.borderpad": 0.4,
        "legend.borderaxespad": 0.8,
    })

    def place_legend(ax=None,
                     position: str | None = None,
                     outside: bool | None = None,
                     fontsize: float | None = None):
        """
        Places the legend on an axis with consistent thesis style.

        Parameters
        ----------
        ax : matplotlib.axes.Axes, optional
            Axis on which to place the legend (default: current axis).
        position : str | None
            'upper right', 'upper left', 'lower right', 'lower left', 'center', 'best'.
            If None, uses the default position passed to apply_thesis_style().
            Spaces/caps tolerated; 'top/bottom' mapped to 'upper/lower'.
        outside : bool | None
            If True, places the legend outside the plot (to the right).
            If None, uses the default value passed to apply_thesis_style().
        fontsize : float | None
            To change the legend font size only in this plot.
        """

        if ax is None:
            ax = plt.gca()

        # Normalize requested position or use global default
        loc_val = _default_loc
        if isinstance(position, str):
            key = position.strip().lower().replace("top", "upper").replace("bottom", "lower")
            loc_map = {
                "upper right": "upper right",
                "upper left":  "upper left",
                "lower right": "lower right",
                "lower left":  "lower left",
                "center":      "center",
                "best":        "best",
            }
            loc_val = loc_map.get(key, _default_loc)

        # outside: if None, inherit from default; otherwise use override
        outside = (legend_outside if outside is None else bool(outside))

        # Build kwargs consistent with rcParams
        legend_kwargs = dict(
            loc=loc_val,
            frameon=True,
            facecolor="white",
            edgecolor=mpl.rcParams["legend.edgecolor"],
            framealpha=1.0,
            fontsize=mpl.rcParams["legend.fontsize"] if fontsize is None else fontsize,
            handlelength=1.8,
            handletextpad=0.6,
            borderpad=0.4,
            fancybox=False,
        )

        if outside:
            legend_kwargs.update({
                "bbox_to_anchor": (1.02, 1.0),
                "borderaxespad": 0.0,
            })

        leg = ax.legend(**legend_kwargs)
        if leg is not None:
            leg.get_frame().set_linewidth(0.8)
            leg.get_frame().set_edgecolor(mpl.rcParams["legend.edgecolor"])
            leg.get_frame().set_facecolor("white")
        return ax

    # Export the helper to the global notebook/script namespace
    globals()["place_legend"] = place_legend
    

## STEP 0

In [ ]:
# EDGAR EXTENSION — IMPORTS AND GLOBAL SETTINGS
RNG = np.random.default_rng(42)

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(12)


In [ ]:
# EDGAR EXTENSION — PATHS AND COLUMN NAMES (PHENO-FREE)

DATA_PATHS = {
    "GSE69914_beta":  "/kaggle/input/gse69914-parquet/GSE69914.parquet",
    "GSE287331_beta": "/kaggle/input/3-gse287331-parquet/GSE287331_clean_imputed.parquet",
    "GSE225845_beta": "/kaggle/input/gse225845-parquet/GSE225845.parquet"
}

BREAST_DATASETS = ["GSE69914", "GSE225845", "GSE287331"]

BETA_ID_COL = "id_tissue"
BETA_LABEL_COL = "label"

NORMAL_LABEL_VALUE = 0

EDGAR_LIST_PATHS = {
    "Blood":     "/kaggle/input/invariant-blood-cpgs-csv/Invariant_Blood_CpGs.csv",
    "Buccal":    "/kaggle/input/invariant-buccal-cpgs-csv/Invariant_Buccal_CpGs.csv",
    "Placenta":  "/kaggle/input/invariant-placenta-cpgs-csv/Invariant_Placenta_CpGs.csv"
}


In [ ]:
# EDGAR EXTENSION — HELPERS (BETA LAZY, NO REORDER)
def scan_beta(dataset: str) -> pl.LazyFrame:
    return pl.scan_parquet(DATA_PATHS[f"{dataset}_beta"])

def infer_cpg_columns(lf: pl.LazyFrame,
                      id_cols: Tuple[str, ...] = ("id_tissue", "label", "sentrix_id", "slide_id", "batch", "group")) -> List[str]:
    cols = lf.collect_schema().names()
    return [c for c in cols if c not in id_cols]

def get_normal_ids_from_beta(dataset: str) -> List[str]:
    lf = scan_beta(dataset)
    ids = (
        lf.filter(pl.col(BETA_LABEL_COL) == NORMAL_LABEL_VALUE)
          .select(BETA_ID_COL)
          .collect()
          .to_series()
          .to_list()
    )
    return ids

def load_matrix_numpy(dataset: str, sample_ids: List[str], cpg_cols: List[str]) -> np.ndarray:
    lf = scan_beta(dataset)
    df = (
        lf.filter(pl.col(BETA_ID_COL).is_in(sample_ids))
          .select([BETA_ID_COL] + cpg_cols)
          .collect()
    )
    return df.select(cpg_cols).to_numpy()


In [ ]:
# EDGAR EXTENSION — COMMON CpGs ACROSS BREAST DATASETS
def get_common_cpgs(datasets: List[str]) -> List[str]:
    sets = []
    for ds in datasets:
        lf = scan_beta(ds)
        sets.append(set(infer_cpg_columns(lf)))
    return sorted(set.intersection(*sets))

COMMON_CPGS = get_common_cpgs(BREAST_DATASETS)
print("COMMON_CPGS across breast datasets:", len(COMMON_CPGS))


In [ ]:
# EDGAR EXTENSION — EDGAR METRIC (P90-P10) AND NON-VARIABLE CALL
def reference_range_p90_p10(X: np.ndarray) -> np.ndarray:
    p90 = np.nanpercentile(X, 90, axis=0)
    p10 = np.nanpercentile(X, 10, axis=0)
    return p90 - p10

def call_nonvariable(ref_range: np.ndarray, threshold: float = 0.05) -> np.ndarray:
    return ref_range < threshold


In [ ]:
# EDGAR EXTENSION — BREAST NON-VARIABLE CpGs (NORMALS-ONLY, POOLED)
def compute_breast_nonvariable_cpgs_pooled(
    datasets: List[str],
    common_cpgs: List[str],
    threshold: float = 0.05,
) -> Tuple[Set[str], pd.DataFrame, pd.DataFrame]:

    X_list = []
    n_normals = {}

    for ds in datasets:
        ids = get_normal_ids_from_beta(ds)
        n_normals[ds] = len(ids)
        X = load_matrix_numpy(ds, ids, cpg_cols=common_cpgs)
        X_list.append(X)

    X_pool = np.vstack(X_list)
    rr = reference_range_p90_p10(X_pool)
    mask = call_nonvariable(rr, threshold=threshold)

    breast_set = set([c for c, m in zip(common_cpgs, mask) if m])

    per_cpg = pd.DataFrame({
        "CpG": common_cpgs,
        "ref_range_p90_p10": rr,
        "nonvariable": mask
    }).sort_values("ref_range_p90_p10")

    meta = pd.DataFrame([{
        "threshold": threshold,
        "n_common_cpgs": len(common_cpgs),
        "n_pooled_normals": int(X_pool.shape[0]),
        "n_nonvariable_breast": len(breast_set),
        **{f"n_normals_{k}": v for k, v in n_normals.items()}
    }])

    return breast_set, per_cpg, meta

BREAST_NONVAR, BREAST_PER_CPG, BREAST_META = compute_breast_nonvariable_cpgs_pooled(
    datasets=BREAST_DATASETS,
    common_cpgs=COMMON_CPGS,
    threshold=0.05
)

BREAST_META


In [ ]:
# EDGAR EXTENSION — LOAD EDGAR LISTS AS CpG SETS
def load_cpg_list(path: str) -> Set[str]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if p.suffix.lower() in [".csv", ".tsv"]:
        df = pd.read_csv(p, sep=None, engine="python")
        if df.shape[1] == 1:
            col = df.columns[0]
            return set(df[col].astype(str).str.strip().tolist())
        for col in ["CpG", "cpg", "cgid", "ID", "probe"]:
            if col in df.columns:
                return set(df[col].astype(str).str.strip().tolist())
        return set(df.iloc[:, 0].astype(str).str.strip().tolist())

    lines = p.read_text().splitlines()
    return set([ln.strip() for ln in lines if ln.strip()])

EDGAR_SETS = {k: load_cpg_list(v) for k, v in EDGAR_LIST_PATHS.items()}
for k, s in EDGAR_SETS.items():
    print(k, len(s))


In [ ]:
# EDGAR EXTENSION — DEFINE UNIVERSE AND TISSUE-SPECIFIC SETS
UNIVERSE = set(COMMON_CPGS)

BREAST_SET   = BREAST_NONVAR & UNIVERSE
BLOOD_SET    = EDGAR_SETS["Blood"] & UNIVERSE
BUCCAL_SET   = EDGAR_SETS["Buccal"] & UNIVERSE
PLACENTA_SET = EDGAR_SETS["Placenta"] & UNIVERSE

# Breast-aware universal non-variable CpGs (extension of Edgar)
UNIVERSAL_BREAST = BREAST_SET & BLOOD_SET & BUCCAL_SET & PLACENTA_SET

print("Universe size:", len(UNIVERSE))
print("Breast:", len(BREAST_SET))
print("Blood:", len(BLOOD_SET))
print("Buccal:", len(BUCCAL_SET))
print("Placenta:", len(PLACENTA_SET))
print("Universal (Blood ∩ Buccal ∩ Placenta ∩ Breast):", len(UNIVERSAL_BREAST))


In [ ]:
# EDGAR EXTENSION — OVERLAP METRICS (PAIRWISE + KEY COUNTS)
def jaccard(a: Set[str], b: Set[str]) -> float:
    return len(a & b) / max(1, len(a | b))

def overlap_table(named_sets: Dict[str, Set[str]]) -> pd.DataFrame:
    names = list(named_sets.keys())
    rows = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            A, B = names[i], names[j]
            s1, s2 = named_sets[A], named_sets[B]
            rows.append({
                "A": A, "B": B,
                "nA": len(s1), "nB": len(s2),
                "intersection": len(s1 & s2),
                "jaccard": jaccard(s1, s2)
            })
    return pd.DataFrame(rows).sort_values("intersection", ascending=False)

NAMED = {
    "Blood": BLOOD_SET,
    "Buccal": BUCCAL_SET,
    "Placenta": PLACENTA_SET,
    "Breast": BREAST_SET
}

PAIRWISE = overlap_table(NAMED)
PAIRWISE


In [ ]:
# EDGAR EXTENSION — INSTALL/IMPORT VENN (RUN ONCE)
try:
    from venn import venn
    VENN_OK = True
except ImportError:
    VENN_OK = False

if not VENN_OK:
    import subprocess
    subprocess.check_call(["pip", "install", "venn"])
    from venn import venn
    VENN_OK = True


In [ ]:
# EDGAR EXTENSION — 4-WAY VENN + KEY INTERSECTIONS + SAVE UNIVERSAL LIST (SEPARATE FIGURES, NO TITLES)
print(f"Blood non-variable CpGs:     {len(BLOOD_SET):,}")
print(f"Buccal non-variable CpGs:    {len(BUCCAL_SET):,}")
print(f"Placenta non-variable CpGs:  {len(PLACENTA_SET):,}")
print(f"Breast non-variable CpGs:    {len(BREAST_SET):,}")

UNIVERSAL_BREAST = BLOOD_SET & BUCCAL_SET & PLACENTA_SET & BREAST_SET
print(f"\n4-way intersection (Blood ∩ Buccal ∩ Placenta ∩ Breast): {len(UNIVERSAL_BREAST):,}")

intersections = {
    "Breast only": len(BREAST_SET - (BLOOD_SET | BUCCAL_SET | PLACENTA_SET)),
    "Blood only": len(BLOOD_SET - (BUCCAL_SET | PLACENTA_SET | BREAST_SET)),
    "Buccal only": len(BUCCAL_SET - (BLOOD_SET | PLACENTA_SET | BREAST_SET)),
    "Placenta only": len(PLACENTA_SET - (BLOOD_SET | BUCCAL_SET | BREAST_SET)),
    "All 4 tissues": len(UNIVERSAL_BREAST),
}

# Figure 1 — 4-way Venn (PDF)
apply_thesis_style(use_tex=True)

fig1, ax1 = plt.subplots(1, 1, figsize=(6.8, 4.5))
ax1.set_aspect("equal", adjustable="box")  # enforce square axes -> consistent size

venn_dict = {
    "Blood": BLOOD_SET,
    "Buccal": BUCCAL_SET,
    "Placenta": PLACENTA_SET,
    "Breast": BREAST_SET,
}
venn(venn_dict, ax=ax1)

# Legend top-right
leg = ax1.get_legend()
if leg is not None:
    leg.set_bbox_to_anchor((1.02, 1.02))
    leg._loc = 2  # upper right

# Do NOT use bbox_inches="tight" if you want identical physical sizes across PDFs
fig1.savefig("/kaggle/working/venn_4tissues_invariants.pdf")
plt.show()

# Figure 2 — Key intersections barplot (PDF)
apply_thesis_style(use_tex=True)

fig2, ax2 = plt.subplots(1, 1, figsize=(6.8, 4.5))

labels = list(intersections.keys())
values = list(intersections.values())

cmap = plt.cm.viridis
colors = cmap(np.linspace(0.15, 0.90, len(labels)))

bars = ax2.barh(
    labels,
    values,
    color=colors,
    linewidth=0.8,
    alpha=0.85
)

ax2.set_xlabel("Number of CpGs")
ax2.grid(axis="x", alpha=0.3, linestyle="--")

# Extend x-axis to a fixed max so the plot layout is stable and labels don't feel cramped
ax2.set_xlim(0, 35000)

maxv = max(values) if len(values) > 0 else 1
for i, (bar, val) in enumerate(zip(bars, values)):
    ax2.text(val + 35000 * 0.01, i, f"{val:,}", va="center")

# Increase left margin (do this once; stable layout)
fig2.subplots_adjust(left=0.42)

fig2.savefig("/kaggle/working/barplot_key_intersections_invariants.pdf")
plt.show()

print("\nSUMMARY OF NON-VARIABLE CpGs")
print(f"Universe (COMMON_CPGS):                           {len(UNIVERSE):,}")
print(f"Blood:                                           {len(BLOOD_SET):,}")
print(f"Buccal:                                          {len(BUCCAL_SET):,}")
print(f"Placenta:                                        {len(PLACENTA_SET):,}")
print(f"Breast:                                          {len(BREAST_SET):,}")
print(f"4-WAY UNIVERSAL (Blood∩Buccal∩Placenta∩Breast):   {len(UNIVERSAL_BREAST):,}")

out_path = "/kaggle/working/Universal_4tissues_invariants.csv"
pl.DataFrame({"CpG_ID": sorted(UNIVERSAL_BREAST)}).write_csv(out_path)
print(f"\nSaved 4-tissue universal invariants to: {out_path}")

print("\nSaved figures:")
print(" - /kaggle/working/venn_4tissues_invariants.pdf")
print(" - /kaggle/working/barplot_key_intersections_invariants.pdf")


## STEP A

In [ ]:
# EDGAR EXTENSION — STEP A: LODO ROBUSTNESS WITHIN BREAST NORMALS (PHENO-FREE, USING label IN BETA)
def compute_breast_nonvariable_lodo(
    train_datasets: List[str],
    test_dataset: str,
    threshold: float = 0.05,
) -> Dict[str, object]:

    train_set, _, _ = compute_breast_nonvariable_cpgs_pooled(
        datasets=train_datasets,
        common_cpgs=COMMON_CPGS,
        threshold=threshold
    )

    ids = get_normal_ids_from_beta(test_dataset)
    X_test = load_matrix_numpy(test_dataset, ids, cpg_cols=COMMON_CPGS)
    rr_test = reference_range_p90_p10(X_test)

    rr_test_df = pd.DataFrame({"CpG": COMMON_CPGS, "rr_test": rr_test})
    rr_test_df["in_train_nonvariable"] = rr_test_df["CpG"].isin(train_set)
    rr_test_df["still_nonvariable_in_test"] = rr_test_df["rr_test"] < threshold

    subset = rr_test_df[rr_test_df["in_train_nonvariable"]]
    retention = float(subset["still_nonvariable_in_test"].mean()) if len(subset) > 0 else np.nan

    summary = {
        "train": ",".join(train_datasets),
        "test": test_dataset,
        "threshold": threshold,
        "n_train_nonvariable": len(train_set),
        "n_test_normals": int(X_test.shape[0]),
        "retention_rate_in_test": retention,
    }

    return {"summary": summary, "per_cpg_test": rr_test_df}

folds = [
    (["GSE225845", "GSE287331"], "GSE69914"),
    (["GSE69914", "GSE287331"], "GSE225845"),
    (["GSE69914", "GSE225845"], "GSE287331"),
]

A_out = []
for train, test in folds:
    res = compute_breast_nonvariable_lodo(train, test, threshold=0.05)
    A_out.append(res["summary"])

A_df = pd.DataFrame(A_out)
A_df


In [ ]:
# EDGAR EXTENSION — STEP A: LODO RETENTION RATES (VIRIDIS)
apply_thesis_style(use_tex=True)

labels = A_df["test"].to_list()
values = A_df["retention_rate_in_test"].to_list()

# Viridis colormap: evenly spaced colors
cmap = plt.cm.viridis
colors = cmap(np.linspace(0.2, 0.8, len(labels)))

plt.figure(figsize=(6.8, 4.5))
plt.bar(labels, values, color=colors, linewidth=0.8)
plt.xlabel("Held-out dataset")
plt.ylabel("Retention rate")
plt.ylim(0, 1.05)


plt.tight_layout()
plt.savefig("/kaggle/working/stepA_lodo_stability.pdf")
plt.show()


## STEP B

In [ ]:
# EDGAR EXTENSION — STEP B (OPTIMIZED): THRESHOLD SENSITIVITY + JACCARD STABILITY (BREAST ONLY)
def breast_threshold_sensitivity_optimized(
    thresholds: List[float],
    folds: List[Tuple[List[str], str]],
    common_cpgs: List[str],
) -> pd.DataFrame:
    """
    Same output as the original Step B, but optimized:
    - For each fold, compute rr = P90-P10 ONCE on the pooled train normals
    - For each threshold, derive the non-variable set via rr < thr (cheap)
    """

    rows = []

    # Precompute rr per fold once (heavy part only 3 times)
    rr_by_fold = []   # list of dict: {"train": [...], "rr": np.ndarray, "n_pooled": int}
    for train, _ in folds:
        X_list = []
        n_pooled = 0
        for ds in train:
            ids = get_normal_ids_from_beta(ds)
            X = load_matrix_numpy(ds, ids, cpg_cols=common_cpgs)
            X_list.append(X)
            n_pooled += int(X.shape[0])

        X_pool = np.vstack(X_list)
        rr = reference_range_p90_p10(X_pool)

        rr_by_fold.append({
            "train": train,
            "train_key": ",".join(train),
            "rr": rr,
            "n_pooled": n_pooled,
        })

    # For each threshold: compute sets cheaply from cached rr, then Jaccard
    for thr in thresholds:
        thr = float(thr)
        sets = []

        for fold_info in rr_by_fold:
            rr = fold_info["rr"]
            train_key = fold_info["train_key"]

            mask = rr < thr
            s = set([c for c, m in zip(common_cpgs, mask) if m])

            sets.append(s)
            rows.append({
                "threshold": thr,
                "train": train_key,
                "n_nonvariable": len(s),
                "J_mean": np.nan,
            })

        # Pairwise Jaccard mean among the 3 train folds
        J = []
        for i in range(len(sets)):
            for j in range(i + 1, len(sets)):
                J.append(jaccard(sets[i], sets[j]))

        rows.append({
            "threshold": thr,
            "train": "PAIRWISE_JACCARD_MEAN",
            "n_nonvariable": np.nan,
            "J_mean": float(np.mean(J)) if len(J) else np.nan
        })

    return pd.DataFrame(rows)

thr_grid = [round(x, 2) for x in np.arange(0.01, 0.101, 0.01)]
B_df = breast_threshold_sensitivity_optimized(
    thresholds=thr_grid,
    folds=folds,
    common_cpgs=COMMON_CPGS
)

B_df.head(12)


In [ ]:
# EDGAR EXTENSION — STEP B: STABILITY AND LIST SIZE VS THRESHOLD (HIGHLIGHT OPTIMAL REGION)
# Prepare data
tmpJ = B_df[B_df["train"] == "PAIRWISE_JACCARD_MEAN"].sort_values("threshold")

tmpN = (
    B_df[B_df["train"] != "PAIRWISE_JACCARD_MEAN"]
      .groupby("threshold")["n_nonvariable"]
      .mean()
      .reset_index()
)

apply_thesis_style(use_tex=True)

# Threshold highlight settings
thr_star = 0.05
thr_low  = 0.04
thr_high = 0.06

fig, ax1 = plt.subplots(figsize=(6.8, 4.5))

# Viridis colors
cmap = plt.cm.viridis
color_j = cmap(0.65)   # Jaccard
color_n = cmap(0.30)   # List size

# Highlight optimal region (vertical band)
ax1.axvspan(thr_low, thr_high, color=cmap(0.85), alpha=0.18, zorder=0)

# Vertical line at Edgar threshold
ax1.axvline(thr_star, color=cmap(0.90), linewidth=2.0, linestyle="--", zorder=1, label="Edgar threshold (0.05)")

# Left axis — Jaccard stability
ax1.plot(
    tmpJ["threshold"],
    tmpJ["J_mean"],
    color=color_j,
    linewidth=2.2,
    label="Stability (mean Jaccard across LODO folds)"
)
ax1.set_xlabel("Threshold (P90–P10 < thr)")
ax1.set_ylabel("Mean Jaccard")
ax1.set_ylim(0, 1.05)

# Right axis — list size
ax2 = ax1.twinx()
ax2.plot(
    tmpN["threshold"],
    tmpN["n_nonvariable"],
    color=color_n,
    linewidth=2.2,
    label="Mean number of non-variable CpGs"
)
ax2.set_ylabel("Mean number of non-variable CpGs")

# Combined legend (top-left)
lines = ax1.get_lines() + ax2.get_lines()
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper left", frameon=True)

plt.tight_layout()
plt.savefig("/kaggle/working/stepB_stability_vs_list_size_highlighted.pdf")
plt.show()


## STEP C

In [ ]:
# EDGAR EXTENSION — STEP C (OPTIMIZED): BETA DISTRIBUTION (NON-VARIABLE VS BACKGROUND), PHENO-FREE
def pooled_normal_betas_two_groups(
    cpgs_nv: List[str],
    cpgs_bg: List[str],
    datasets: List[str],
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Loads pooled normal betas once per dataset for (nv ∪ bg),
    then splits into NV and BG matrices without re-loading.
    """
    cpgs_nv = list(cpgs_nv)
    cpgs_bg = list(cpgs_bg)
    all_cpgs = list(dict.fromkeys(cpgs_nv + cpgs_bg))  # keep order, unique

    nv_idx = np.array([i for i, c in enumerate(all_cpgs) if c in set(cpgs_nv)], dtype=int)
    bg_idx = np.array([i for i, c in enumerate(all_cpgs) if c in set(cpgs_bg)], dtype=int)

    X_nv_list, X_bg_list = [], []
    for ds in datasets:
        ids = get_normal_ids_from_beta(ds)
        X_all = load_matrix_numpy(ds, ids, cpg_cols=all_cpgs)  # ONE LOAD per dataset
        X_nv_list.append(X_all[:, nv_idx])
        X_bg_list.append(X_all[:, bg_idx])

    return np.vstack(X_nv_list), np.vstack(X_bg_list)

BREAST_FINAL = BREAST_SET

background = sorted(list(UNIVERSE - BREAST_FINAL))
bg_sub = RNG.choice(background, size=min(20000, len(background)), replace=False).tolist()

# Load once per dataset, split into two matrices
X_nv, X_bg = pooled_normal_betas_two_groups(
    cpgs_nv=sorted(list(BREAST_FINAL)),
    cpgs_bg=bg_sub,
    datasets=BREAST_DATASETS
)

# Flatten + drop NaNs
nv_vals = X_nv.ravel()
bg_vals = X_bg.ravel()
nv_vals = nv_vals[~np.isnan(nv_vals)]
bg_vals = bg_vals[~np.isnan(bg_vals)]

apply_thesis_style(use_tex=True)

cmap = plt.cm.viridis
color_nv = cmap(0.25)   # non-variable CpGs
color_bg = cmap(0.75)   # background

plt.figure(figsize=(6.8, 4.5))

plt.hist(
    nv_vals,
    bins=100,
    density=True,
    color=color_nv,
    alpha=0.85,
    label="Breast non-variable"
)

plt.hist(
    bg_vals,
    bins=100,
    density=True,
    color=color_bg,
    alpha=0.55,
    label="Background (subsample)"
)

plt.xlabel(r"$\beta$ value (Normals pooled)")
plt.ylabel("Density")

place_legend(position="upper right", outside=False)
plt.tight_layout()

plt.savefig(
    "/kaggle/working/stepC_beta_distribution_nv_vs_background.pdf",
    bbox_inches="tight"
)
plt.show()



## STEP D

In [ ]:
# EDGAR EXTENSION — STEP D (OPTIMIZED): BASELINE STABILITY IN NORMALS (RR ECDF + SUMMARY TABLE)
thr_star = 0.05

# Define groups (all within UNIVERSE)
BREAST_NV = set(BREAST_SET) & set(UNIVERSE)
UNIVERSAL_4WAY = (set(BLOOD_SET) & set(BUCCAL_SET) & set(PLACENTA_SET) & set(BREAST_SET)) & set(UNIVERSE)
BREAST_ONLY = (set(BREAST_SET) - (set(BLOOD_SET) | set(BUCCAL_SET) | set(PLACENTA_SET))) & set(UNIVERSE)

# Background: sample CpGs not in breast non-variable
background_pool = sorted(list(set(UNIVERSE) - BREAST_NV))
bg_size = min(20000, len(background_pool))  # keep it lightweight but representative
BG = set(RNG.choice(background_pool, size=bg_size, replace=False).tolist())

# Build a single CpG universe to load ONCE per dataset
# (union of all groups; keep order stable)
all_cpgs = list(dict.fromkeys(
    sorted(list(BREAST_NV)) +
    sorted(list(BREAST_ONLY)) +
    sorted(list(UNIVERSAL_4WAY)) +
    sorted(list(BG))
))

# Indices for each group within all_cpgs (cheap slicing later)
idx_map = {c: i for i, c in enumerate(all_cpgs)}
idx_breast_nv = np.array([idx_map[c] for c in BREAST_NV if c in idx_map], dtype=int)
idx_breast_only = np.array([idx_map[c] for c in BREAST_ONLY if c in idx_map], dtype=int)
idx_univ_4way = np.array([idx_map[c] for c in UNIVERSAL_4WAY if c in idx_map], dtype=int)
idx_bg = np.array([idx_map[c] for c in BG if c in idx_map], dtype=int)

# Load pooled normals ONCE per dataset for all_cpgs
X_list = []
n_pool = 0
for ds in BREAST_DATASETS:
    ids = get_normal_ids_from_beta(ds)  # label-based, pheno-free
    X = load_matrix_numpy(ds, ids, cpg_cols=all_cpgs)
    X_list.append(X)
    n_pool += int(X.shape[0])

X_pool = np.vstack(X_list)

# Compute rr once for all CpGs
rr_all = reference_range_p90_p10(X_pool)  # length = len(all_cpgs)

# Helper: summarize rr for a group
def summarize_group(rr_vec: np.ndarray, name: str) -> dict:
    rr_vec = rr_vec[~np.isnan(rr_vec)]
    return {
        "group": name,
        "n_cpgs": int(rr_vec.size),
        "median_rr": float(np.median(rr_vec)) if rr_vec.size else np.nan,
        "mean_rr": float(np.mean(rr_vec)) if rr_vec.size else np.nan,
        "p95_rr": float(np.percentile(rr_vec, 95)) if rr_vec.size else np.nan,
        "pct_below_0.05": float(np.mean(rr_vec < thr_star)) if rr_vec.size else np.nan,
    }

rr_breast_nv = rr_all[idx_breast_nv]
rr_breast_only = rr_all[idx_breast_only]
rr_univ_4way = rr_all[idx_univ_4way]
rr_bg = rr_all[idx_bg]

summary_rows = [
    summarize_group(rr_univ_4way, "Universal (4-way)"),
    summarize_group(rr_breast_nv, "Breast non-variable"),
    summarize_group(rr_breast_only, "Breast-only non-variable"),
    summarize_group(rr_bg, "Background (random)"),
]
D_df = pd.DataFrame(summary_rows)
print(f"Pooled normal samples used (total): {n_pool:,}")
D_df

# ECDF plot (clean + comparable)
def ecdf(x: np.ndarray):
    x = x[~np.isnan(x)]
    x = np.sort(x)
    y = np.linspace(0, 1, len(x), endpoint=True) if len(x) else np.array([])
    return x, y

apply_thesis_style(use_tex=True)

fig, ax = plt.subplots(figsize=(6.8, 4.5))

cmap = plt.cm.viridis
colors = {
    "Universal (4-way)": cmap(0.85),
    "Breast non-variable": cmap(0.55),
    "Breast-only non-variable": cmap(0.30),
    "Background (random)": cmap(0.10),
}

series = [
    ("Universal (4-way)", rr_univ_4way),
    ("Breast non-variable", rr_breast_nv),
    ("Breast-only non-variable", rr_breast_only),
    ("Background (random)", rr_bg),
]

for name, rr_vec in series:
    x, y = ecdf(rr_vec)
    ax.plot(x, y, linewidth=2.0, color=colors[name], label=name)

# Edgar threshold reference
ax.axvline(thr_star, linestyle="--", linewidth=1.8, color=cmap(0.95), label="Threshold 0.05")

ax.set_xlabel(r"Within-normal variability $rr = P90 - P10$")
ax.set_ylabel("ECDF")
ax.set_xlim(0, min(0.20, np.nanpercentile(rr_all, 99)))  # keep focus on informative range
ax.set_ylim(0, 1.02)

# Legend top-left
ax.legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.savefig("/kaggle/working/stepD_baseline_rr_ecdf.pdf")
plt.show()
